In [ ]:
import numpy as np
import scipy
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, permutation_test_score
from sklearn.metrics import classification_report, balanced_accuracy_score, accuracy_score

data_path = Path("") # Insert path to the folder containing the .npy files

In [ ]:
def load_user_data():
    centroid_mat = scipy.io.loadmat(data_path / "<user_id>.mat")
    raw_centroids = centroid_mat["run_centroids"][0, :, :, :]
    run_centroids = np.concatenate([raw_centroids[:, 0, :, :], raw_centroids[:, 1, :, :]], axis=0)
    print("Loaded centroid data shape:", run_centroids.shape)

    cluster_labels = np.load(data_path / 'cluster_labels.npy')[0, :] 
    cluster_labels = np.concatenate([cluster_labels, cluster_labels], axis=0)
    cluster_labels = cluster_labels.astype(int) 
    print("Loaded cluster labels shape:", cluster_labels.shape)

    return run_centroids, cluster_labels

def align_eigenvector_signs(X_raw, num_features_per_vector):
    """
    Enforces sign consistency across eigenvectors to resolve the v == -v ambiguity.
    Flips each eigenvector so its maximum absolute value is always positive.
    """
    X_aligned = np.copy(X_raw)
    num_vectors = X_raw.shape[1] // num_features_per_vector
    
    for i in range(len(X_aligned)):
        for j in range(num_vectors):
            start_idx = j * num_features_per_vector
            end_idx = start_idx + num_features_per_vector
            vec = X_aligned[i, start_idx:end_idx]
            
            max_val_idx = np.argmax(np.abs(vec))
            X_aligned[i, start_idx:end_idx] = vec * np.sign(vec[max_val_idx])
            
    return X_aligned

def get_eigen_details(cov):
    vals, vecs = np.linalg.eigh(cov)
    idx = np.argsort(vals)[::-1]
    return vals[idx], vecs[:, idx]

In [ ]:
# --- 1. Load Complete Dataset ---
data, labels = load_user_data()
labels = np.array(labels)

# 3 eigenvectors * 13 channels = 39 features total
CHANNELS_PER_VECTOR = 13 
X_raw = np.zeros((len(data), 39)) 

# --- 2. Feature Extraction ---
for i, cov in enumerate(data):
    eigenvals, eigenvecs = get_eigen_details(cov)
    X_raw[i] = eigenvecs[:, :3].flatten() 

# Resolve the v == -v geometric alignment ambiguity
X_all = align_eigenvector_signs(X_raw, num_features_per_vector=CHANNELS_PER_VECTOR)

# --- 3. Noise Filter Layer (Drop Cluster -1) ---
clean_mask = (labels != -1)
X = X_all[clean_mask]
labels_clean = labels[clean_mask]

# Map remaining labels: 0 & 1 -> Stable (0), 2 & 3 -> Emergent (1)
y = np.zeros(len(labels_clean))
for i, label in enumerate(labels_clean):
    if label in [0, 1]:
        y[i] = 0                  # Stable State
    elif label in [2, 3]:
        y[i] = 1                  # Emergent State

print("Noise-Filtered Dataset Shapes:")
print(f"Clean Features matrix X: {X.shape}")
print(f"Clean Target vector y:   {y.shape}")
print(f"Retained Organic Clusters: {np.unique(labels_clean)}\n")

# Base configuration parameters for multi-seed robustness tracking
classifier = LinearDiscriminantAnalysis()
SEEDS = [19, 42, 87, 101, 144, 256, 312, 512, 777, 999]
NUM_PERMUTATIONS = 500 

state_names = ['Stable', 'Emergent']
unique_clusters_clean = np.sort(np.unique(labels_clean))

# Accumulator metrics for multi-seed averaging
part1_global_accs = []
part1_balanced_accs = []
part1_pvalues = []

# Tracker matrix to map cluster recognition rates across seeds
# Rows: Organic clusters (0, 1, 2, 3) | Columns: Predicted states (Stable=0, Emergent=1)
cluster_mapping_counts = np.zeros((len(unique_clusters_clean), len(state_names)))

# =============================================================================
# MULTI-SEED STRATIFIED CROSS-VALIDATION PIPELINE
# =============================================================================
print("================ RUNNING MULTI-SEED STATE-LEVEL VALIDATION ================")
for seed in SEEDS:
    cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    
    for train_idx, test_idx in cv_strat.split(X, y):
        classifier.fit(X[train_idx], y[train_idx])
        preds = classifier.predict(X[test_idx])
        
        part1_global_accs.append(accuracy_score(y[test_idx], preds))
        part1_balanced_accs.append(balanced_accuracy_score(y[test_idx], preds))
        
        # Log out-of-fold predictions matched against their source organic clusters
        for test_i, pred_state in zip(test_idx, preds):
            source_cluster = labels_clean[test_i]
            cluster_row_idx = np.where(unique_clusters_clean == source_cluster)[0][0]
            cluster_mapping_counts[cluster_row_idx, int(pred_state)] += 1
            
    # Track significance threshold bounds per partition
    _, _, p_val = permutation_test_score(
        classifier, X, y, cv=cv_strat, scoring='balanced_accuracy', 
        n_permutations=NUM_PERMUTATIONS, n_jobs=-1, random_state=seed
    )
    part1_pvalues.append(p_val)

# Normalize the prediction tracker array to get average values per seed
cluster_mapping_avg = cluster_mapping_counts / len(SEEDS)

# =============================================================================
# GLOBAL PERFORMANCE PRINT OUT
# =============================================================================
print("\n----------------------------------------------------------------")
print(f"Overall Global Accuracy across {len(SEEDS)} seeds:   {np.mean(part1_global_accs) * 100:.2f}%")
print(f"Overall Balanced Accuracy across {len(SEEDS)} seeds: {np.mean(part1_balanced_accs) * 100:.2f}%")
print(f"Average Permutation Significance p-value:       {np.mean(part1_pvalues):.4f}")
print("----------------------------------------------------------------\n")

# =============================================================================
# GRANULAR ORGANIC CLUSTER RECOGNITION REPORT
# =============================================================================
print("================ GRANULAR CLUSTER RECOGNITION RATES ================")
for c_idx, cluster in enumerate(unique_clusters_clean):
    total_cluster_samples = np.sum(labels_clean == cluster)
    stable_hits = cluster_mapping_avg[c_idx, 0]
    emergent_hits = cluster_mapping_avg[c_idx, 1]
    
    stable_pct = (stable_hits / total_cluster_samples) * 100
    emergent_pct = (emergent_hits / total_cluster_samples) * 100
    
    print(f"Cluster {cluster} (Total Support: {total_cluster_samples} samples):")
    print(f"  Identified as 'Stable':   {int(np.round(stable_hits))} samples - seed ({stable_pct:.2f}%)")
    print(f"  Identified as 'Emergent': {int(np.round(emergent_hits))} samples - seed ({emergent_pct:.2f}%)")
print("====================================================================")